# COMP8240 — Clean Pima + RTML Replication

This notebook is a **separate, original-style reproduction** of the Pima + RTML XGBoost/ADASYN experiment. It retains the successful code and recorded outputs from the student’s Colab run. It is **not** an exact reproduction of all original author settings, and it is **not a leakage-free clinical validation**.

**Source research:** Tasin et al. (2023), *Diabetes prediction using machine learning and explainable AI techniques*, https://doi.org/10.1049/htl2.12039. Original authors’ code: https://github.com/tansin-nabil/Diabetes-Prediction-Using-Machine-Learning.

**Required files:** `diabetes.csv` (Pima data) and `RTML without Insulin.xlsx` (from the authors’ repository). In Google Colab, open the Files sidebar and upload both files to `/content/` before running **Runtime → Run all**. Do not commit sensitive datasets or personal identifiers.

**Data provenance:** 768 Pima records and 109 RTML records. The RTML dataset contains 109 missing insulin measurements; the Pima dataset contains 374 zero insulin measurements.

**Methodological caveats:** (1) Pima mean imputation is fitted before the final classification split; (2) the RTML insulin regressor is trained using Pima data before that split; (3) the regressor’s held-out MAE includes some mean-imputed Pima insulin targets; (4) default classifier hyperparameters may differ from the original study. These choices can affect estimates and permit information leakage. Do not use these models for medical decisions.

**Reproducibility:** Saved outputs reflect the prior Colab environment (XGBoost 3.4.1, scikit-learn 1.6.1, imbalanced-learn 0.14.2); rerun to verify in your environment.


In [2]:

import pandas as pd

Pima_dataset = pd.read_csv('/content/diabetes.csv')

print("Dataset shape:", Pima_dataset.shape)
print(Pima_dataset.head())


Dataset shape: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [3]:

import pandas as pd

rtml = pd.read_excel('/content/RTML without Insulin.xlsx')

print("RTML shape:", rtml.shape)
print("Columns:", rtml.columns.tolist())
print(rtml.head())


RTML shape: (109, 8)
Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age', 'Outcome']
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin        BMI  \
0            5     88.2          106.0            4.8      NaN  14.794213   
1            6     73.8           74.0            7.5      NaN  24.851410   
2            1     86.4           87.0           18.4      NaN  41.621307   
3            1     97.2           71.0           15.0      NaN  22.608427   
4            5     90.0           96.0            7.6      NaN  19.154528   

   Age  Outcome  
0   50        0  
1   50        0  
2   30        0  
3   28        0  
4   50        0  


In [4]:

print("RTML missing values:")
print(rtml.isnull().sum())

print("\nRTML duplicate rows:")
print(rtml.duplicated().sum())

print("\nRTML class distribution:")
print(rtml["Outcome"].value_counts())

print("\nRTML data types:")
print(rtml.dtypes)


RTML missing values:
Pregnancies        0
Glucose            0
BloodPressure      0
SkinThickness      0
Insulin          109
BMI                0
Age                0
Outcome            0
dtype: int64

RTML duplicate rows:
0

RTML class distribution:
Outcome
0    86
1    23
Name: count, dtype: int64

RTML data types:
Pregnancies        int64
Glucose          float64
BloodPressure    float64
SkinThickness    float64
Insulin          float64
BMI              float64
Age                int64
Outcome            int64
dtype: object


In [5]:

print("Pima insulin summary:")
print(Pima_dataset["Insulin"].describe())

print("\nPima insulin zero values:")
print((Pima_dataset["Insulin"] == 0).sum())

print("\nRTML insulin missing values:")
print(rtml["Insulin"].isna().sum())


Pima insulin summary:
count    768.000000
mean      79.799479
std      115.244002
min        0.000000
25%        0.000000
50%       30.500000
75%      127.250000
max      846.000000
Name: Insulin, dtype: float64

Pima insulin zero values:
374

RTML insulin missing values:
109


In [6]:

print("Pima shape:", Pima_dataset.shape)
print("RTML shape:", rtml.shape)

print("\nPima columns:")
print(Pima_dataset.columns.tolist())

print("\nRTML columns:")
print(rtml.columns.tolist())

print("\nShared columns:")
print(sorted(set(Pima_dataset.columns) & set(rtml.columns)))


Pima shape: (768, 9)
RTML shape: (109, 8)

Pima columns:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

RTML columns:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age', 'Outcome']

Shared columns:
['Age', 'BMI', 'BloodPressure', 'Glucose', 'Insulin', 'Outcome', 'Pregnancies', 'SkinThickness']


## Original-style preprocessing

The next cell replaces zeros in five Pima medical features with missing values and fits mean imputation on **all** Pima rows. This follows the exploratory original-style workflow but is **not leakage-controlled**.


In [7]:

from sklearn.impute import SimpleImputer

# Keep the original datasets unchanged
pima_rep = Pima_dataset.copy()
rtml_rep = rtml.copy()

# Align the columns
pima_rep = pima_rep.drop(
    columns=["DiabetesPedigreeFunction"]
)

# Identify Pima medical measurements
medical_cols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

# Treat zero measurements as missing
pima_rep[medical_cols] = (
    pima_rep[medical_cols].replace(0, float("nan"))
)

# Apply the original-style mean imputation
imputer = SimpleImputer(strategy="mean")
pima_rep[medical_cols] = imputer.fit_transform(
    pima_rep[medical_cols]
)

print("Prepared Pima shape:", pima_rep.shape)
print("\nMissing values after preparation:")
print(pima_rep.isna().sum())
print("\nMean insulin:", round(pima_rep["Insulin"].mean(), 2))


Prepared Pima shape: (768, 8)

Missing values after preparation:
Pregnancies      0
Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
Age              0
Outcome          0
dtype: int64

Mean insulin: 155.55


In [8]:

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Input features for insulin prediction
features = [
    "Pregnancies", "Glucose", "BloodPressure",
    "SkinThickness", "BMI", "Age"
]

X = pima_rep[features]
y = pima_rep["Insulin"]

# Hold out some Pima records to check prediction error
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train insulin prediction model
insulin_model = XGBRegressor(
    max_depth=10,
    learning_rate=0.1,
    n_estimators=100,
    objective="reg:squarederror",
    random_state=42
)

insulin_model.fit(X_train, y_train)

# Evaluate on held-out Pima records
test_predictions = insulin_model.predict(X_test)

print("Insulin prediction MAE:",
      round(mean_absolute_error(y_test, test_predictions), 2))

# Train on all prepared Pima records before RTML prediction
insulin_model.fit(X, y)

# Estimate the missing RTML insulin measurements
rtml_rep["Insulin"] = insulin_model.predict(
    rtml_rep[features]
)

print("\nRTML shape:", rtml_rep.shape)
print("Missing insulin values:",
      rtml_rep["Insulin"].isna().sum())
print("Predicted insulin summary:")
print(rtml_rep["Insulin"].describe())


Insulin prediction MAE: 43.8

RTML shape: (109, 8)
Missing insulin values: 0
Predicted insulin summary:
count    109.000000
mean     136.789688
std       50.675297
min       43.864151
25%      104.902451
50%      124.243423
75%      159.981735
max      298.197601
Name: Insulin, dtype: float64


In [9]:

# Combine prepared Pima and RTML datasets
combined_rep = pd.concat(
    [pima_rep, rtml_rep],
    ignore_index=True
)

print("Combined dataset shape:", combined_rep.shape)

print("\nMissing values:")
print(combined_rep.isna().sum())

print("\nClass distribution:")
print(combined_rep["Outcome"].value_counts())

print("\nDuplicate rows:")
print(combined_rep.duplicated().sum())

print("\nFirst five records:")
print(combined_rep.head())


Combined dataset shape: (877, 8)

Missing values:
Pregnancies      0
Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
Age              0
Outcome          0
dtype: int64

Class distribution:
Outcome
0    586
1    291
Name: count, dtype: int64

Duplicate rows:
0

First five records:
   Pregnancies  Glucose  BloodPressure  SkinThickness     Insulin   BMI  Age  \
0            6    148.0           72.0       35.00000  155.548223  33.6   50   
1            1     85.0           66.0       29.00000  155.548223  26.6   31   
2            8    183.0           64.0       29.15342  155.548223  23.3   32   
3            1     89.0           66.0       23.00000   94.000000  28.1   21   
4            0    137.0           40.0       35.00000  168.000000  43.1   33   

   Outcome  
0        1  
1        0  
2        1  
3        0  
4        1  


## Classification with ADASYN

ADASYN is applied **only to training rows**. The held-out test set remains unmodified. The saved results below are from a previous run and should be checked by rerunning all cells.


In [10]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

# Separate input features and target
X = combined_rep.drop(columns=["Outcome"])
y = combined_rep["Outcome"]

# Create an 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

# Apply ADASYN only to the training data
adasyn = ADASYN(random_state=0)
X_train_balanced, y_train_balanced = adasyn.fit_resample(
    X_train, y_train
)

# Train XGBoost
model_rep = XGBClassifier(
    random_state=0,
    eval_metric="logloss"
)
model_rep.fit(X_train_balanced, y_train_balanced)

# Evaluate on untouched test data
y_pred = model_rep.predict(X_test)
y_prob = model_rep.predict_proba(X_test)[:, 1]

print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Balanced training records:", len(X_train_balanced))

print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))


Training records: 701
Test records: 176
Balanced training records: 894

Accuracy: 0.7443
F1-score: 0.6087
ROC-AUC: 0.7949

Confusion matrix:
[[96 22]
 [23 35]]

Classification report:
              precision    recall  f1-score   support

           0       0.81      0.81      0.81       118
           1       0.61      0.60      0.61        58

    accuracy                           0.74       176
   macro avg       0.71      0.71      0.71       176
weighted avg       0.74      0.74      0.74       176



In [11]:

import xgboost
import sklearn
import imblearn

print("XGBoost version:", xgboost.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Imbalanced-learn version:", imblearn.__version__)

print("\nXGBoost parameters:")
print(model_rep.get_params())

print("\nTraining class counts before ADASYN:")
print(y_train.value_counts())

print("\nTraining class counts after ADASYN:")
print(y_train_balanced.value_counts())


XGBoost version: 3.4.1
Scikit-learn version: 1.6.1
Imbalanced-learn version: 0.14.2

XGBoost parameters:
{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': None, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': None, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 0, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_paramet

In [ ]:

import json

replication_summary = {
    "experiment": "Pima + RTML, XGBoost + ADASYN",
    "dataset_rows": 877,
    "train_rows": 701,
    "test_rows": 176,
    "adasyn_train_rows": 894,
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "f1_score": round(float(f1_score(y_test, y_pred)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
    "xgboost_version": xgboost.__version__,
    "sklearn_version": sklearn.__version__,
    "imblearn_version": imblearn.__version__,
    "random_state": 0,
    "limitations": [
        "Original-style preprocessing before classification split",
        "Insulin values estimated using XGBoost regression",
        "Exact equivalence to authors' implementation not established"
    ]
}

with open(
    "/content/COMP8240_replication_summary.json", "w"
) as f:
    json.dump(replication_summary, f, indent=2)

print("Replication summary saved successfully.")
